# RT Paper 3 Public Solidification Runner

This notebook runs the next publication-facing Paper 3 batch:

- one public benchmark on Qwen
- one non-Qwen 3B hard-set probe
- one dense crossover sweep on the hard set plus the public benchmark
- pairwise reports for the codec families
- optional artifact publication back into `artifacts/paper3`


In [ ]:
REPO_URL = "https://github.com/SteveMama/rt-geometry-memory.git"
REPO_DIR = "/content/rt-geometry-memory"
BATCH_PREFIX = "paper3_public_v1"
PUBLIC_BENCHMARK_NAME = "longmemeval_s_cleaned"
PUBLIC_BENCHMARK_FORMAT = "longmemeval"  # recommended first benchmark
PUBLIC_BENCHMARK_SOURCE = "/content/longmemeval_s_cleaned.json"
PUBLIC_BENCHMARK_JSONL = "/content/rt-geometry-memory/benchmarks/public_benchmark_normalized.jsonl"
QWEN_MODEL = "qwen25_15b"
NONQWEN_MODEL = "llama32_3b"
TINY_PUBLIC_LIMIT = 2
TINY_PUBLIC_STRIDE = 16
TINY_PUBLIC_MAX_TARGETS = 12
MEDIUM_PUBLIC_LIMIT = 10
MEDIUM_PUBLIC_STRIDE = 8
MEDIUM_PUBLIC_MAX_TARGETS = 24
FULL_PUBLIC_LIMIT = 25
FULL_PUBLIC_STRIDE = 8
FULL_PUBLIC_MAX_TARGETS = 32
FULL_CROSSOVER_LIMIT = 20
FULL_CROSSOVER_STRIDE = 10
FULL_CROSSOVER_MAX_TARGETS = 24


In [ ]:
%cd /content
!rm -rf $REPO_DIR
!git clone $REPO_URL $REPO_DIR
%cd $REPO_DIR
!bash scripts/colab_setup.sh


## Optional Hugging Face login

Use this if you need higher Hub rate limits or gated model access for `llama32_3b`.

In [ ]:
# import os
# os.environ["HF_TOKEN"] = "..."
# from huggingface_hub import login
# login(token=os.environ["HF_TOKEN"])


## Download the recommended public benchmark

Default choice: `LongMemEval-S cleaned`. If you already have a normalized JSONL, skip this and set `PUBLIC_BENCHMARK_FORMAT = "normalized"` plus `PUBLIC_BENCHMARK_SOURCE` to that file.

In [ ]:
!python scripts/download_public_benchmark.py \
  --benchmark $PUBLIC_BENCHMARK_NAME \
  --output $PUBLIC_BENCHMARK_SOURCE


## Prepare the public benchmark JSONL

The default path below converts `LongMemEval-S cleaned` into the project JSONL format. If your benchmark is already normalized, set `PUBLIC_BENCHMARK_FORMAT = "normalized"` and point `PUBLIC_BENCHMARK_SOURCE` at that JSONL file.

In [ ]:
!python scripts/prepare_public_benchmark_jsonl.py \
  --format $PUBLIC_BENCHMARK_FORMAT \
  --input $PUBLIC_BENCHMARK_SOURCE \
  --output $PUBLIC_BENCHMARK_JSONL \
  --family public_benchmark


## Tiny smoke run

Use this first. It confirms the benchmark path, model loading, and bounded target-turn sampling without launching a publication-scale batch.

In [ ]:
tiny_cmd = f"""python -m paper3_codec.study \
  --study-name {BATCH_PREFIX}_tiny_public \
  --model-keys {QWEN_MODEL} \
  --input-path {PUBLIC_BENCHMARK_JSONL} \
  --families '' \
  --limit-conversations {TINY_PUBLIC_LIMIT} \
  --budgets 0.35 \
  --policies uniform,geometry,geometry_keep_compress_drop \
  --recent-window 2 \
  --min-history 50 \
  --max-input-tokens 512 \
  --segment-span 2 \
  --target-turn-stride {TINY_PUBLIC_STRIDE} \
  --max-target-turns {TINY_PUBLIC_MAX_TARGETS}"""
print(tiny_cmd)
!{tiny_cmd}
!bash scripts/run_paper3_pairwise_report.sh results/paper3/studies/{BATCH_PREFIX}_tiny_public


## Medium public-benchmark subset

Run this after the tiny smoke succeeds. This is the right first result-bearing benchmark pass on Qwen.

In [ ]:
medium_cmd = f"bash scripts/run_paper3_public_benchmark.sh {BATCH_PREFIX}_medium_public {PUBLIC_BENCHMARK_JSONL} {QWEN_MODEL} 0.20,0.35,0.50 uniform,semantic,geometry,geometry_segment_actions,geometry_keep_compress_drop {MEDIUM_PUBLIC_LIMIT} {MEDIUM_PUBLIC_STRIDE} {MEDIUM_PUBLIC_MAX_TARGETS}"
print(medium_cmd)
!{medium_cmd}


## Full bounded solidification batch

Use this only after the medium public run succeeds. This launches the full public benchmark, non-Qwen 3B probe, and crossover sweep with bounded defaults.

In [ ]:
full_public_cmd = f"bash scripts/run_paper3_public_benchmark.sh {BATCH_PREFIX}_public_benchmark {PUBLIC_BENCHMARK_JSONL} {QWEN_MODEL} 0.20,0.35,0.50 uniform,semantic,geometry,geometry_segment_actions,geometry_keep_compress_drop {FULL_PUBLIC_LIMIT} {FULL_PUBLIC_STRIDE} {FULL_PUBLIC_MAX_TARGETS}"
full_nonqwen_cmd = f"bash scripts/run_paper3_nonqwen_3b_probe.sh {BATCH_PREFIX}_nonqwen_3b {NONQWEN_MODEL} 0.20,0.35,0.50 paper1_geometry/assets/paper2_behavior_stress_conversations.jsonl uniform,geometry,geometry_keep_compress_drop 9 1 64"
full_crossover_cmd = f"bash scripts/run_paper3_crossover_sweep.sh {BATCH_PREFIX}_crossover {PUBLIC_BENCHMARK_JSONL} {QWEN_MODEL},{NONQWEN_MODEL} 0.24,0.32,0.35,0.42,0.50 uniform,semantic,geometry,geometry_keep_compress_drop {FULL_CROSSOVER_LIMIT} {FULL_CROSSOVER_STRIDE} {FULL_CROSSOVER_MAX_TARGETS}"
print(full_public_cmd)
print(full_nonqwen_cmd)
print(full_crossover_cmd)
!{full_public_cmd}
!{full_nonqwen_cmd}
!{full_crossover_cmd}


## Inspect the main reports

In [ ]:
!sed -n '1,220p' results/paper3/studies/{BATCH_PREFIX}_tiny_public/study_report.md
!sed -n '1,220p' results/paper3/studies/{BATCH_PREFIX}_tiny_public/pairwise_report.md
!sed -n '1,220p' results/paper3/studies/{BATCH_PREFIX}_medium_public/study_report.md
!sed -n '1,220p' results/paper3/studies/{BATCH_PREFIX}_public_benchmark/study_report.md
!sed -n '1,220p' results/paper3/studies/{BATCH_PREFIX}_public_benchmark/pairwise_report.md
!sed -n '1,220p' results/paper3/studies/{BATCH_PREFIX}_nonqwen_3b/study_report.md
!sed -n '1,220p' results/paper3/studies/{BATCH_PREFIX}_crossover/study_report.md
!sed -n '1,220p' results/paper3/studies/{BATCH_PREFIX}_crossover/pairwise_report.md


## Publish tracked artifacts

In [ ]:
!bash scripts/publish_artifact.sh --paper paper3 --source results/paper3/studies/{BATCH_PREFIX}_public_benchmark --name {BATCH_PREFIX}_public_benchmark
!bash scripts/publish_artifact.sh --paper paper3 --source results/paper3/studies/{BATCH_PREFIX}_nonqwen_3b --name {BATCH_PREFIX}_nonqwen_3b
!bash scripts/publish_artifact.sh --paper paper3 --source results/paper3/studies/{BATCH_PREFIX}_crossover --name {BATCH_PREFIX}_crossover


## Optional push from Colab

In [ ]:
# !git status --short
# !git add artifacts/paper3
# !git commit -m "Add Paper 3 public-solidification artifacts"
# !git push
